In [1]:
from google.colab import drive  # comment out when running locally
import numpy as np
import pandas as pd
import os
from pathlib import Path
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from itertools import chain
from itertools import combinations

In [ ]:
# Colab: mount Google Drive and read the data from it
drive.mount("/content/drive", force_remount=True)
BASE = Path("/content/drive/MyDrive/bt4222/JDsearch")

# Local: comment out the two Colab lines above (and the drive import) and uncomment this line
# BASE = Path("JDsearch")

PRODUCT_TXT = BASE / "product_meta_data.txt"
USER_TXT = BASE / "user_behavior_data.txt"

# Load user behaviour data (one row per query session)
user_df = pd.read_csv(USER_TXT, sep="\t", dtype=str, low_memory=False)
print("user_behaviour shape:", user_df.shape)

In [3]:
user_df[user_df['candidate_label_list'].apply(len) == 0]

,query,candidate_wid_list,candidate_label_list,history_qry_list,history_wid_list,history_type_list,history_time_list


In [4]:
# --- HELPER FUNCTIONS ---
def safe_split(val, sep='_'):
    if pd.isna(val) or str(val).strip() == '':
        return []
    return str(val).split(sep)

def safe_split_numeric(val, sep='_'):
    lst = safe_split(val, sep)
    return [float(x) for x in lst]

# 1. PARSING THE DATAFRAME
user_df['candidate_wid_list_parsed'] = user_df['candidate_wid_list'].apply(safe_split)  # ids kept as strings to match history_wid_list
user_df['candidate_label_list_parsed'] = user_df['candidate_label_list'].apply(safe_split_numeric)
user_df['history_qry_list_parsed'] = user_df['history_qry_list'].apply(safe_split)
user_df['history_wid_list_parsed'] = user_df['history_wid_list'].apply(safe_split)
user_df['history_type_list_parsed'] = user_df['history_type_list'].apply(safe_split)
user_df['history_time_list_parsed'] = user_df['history_time_list'].apply(safe_split_numeric)

In [5]:
user_df.head()

,query,candidate_wid_list,candidate_label_list,history_qry_list,history_wid_list,history_type_list,history_time_list,candidate_wid_list_parsed,candidate_label_list_parsed,history_qry_list_parsed,history_wid_list_parsed,history_type_list_parsed,history_time_list_parsed
0,63995226602574196100244625575117,36098226_17729172_34921664_68317377_693764_360...,1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,-1_-1_-1_68784030_36547097602574193663289_95...,9227588_52788392_4585436_87422901_57364733_210...,ORD_ORD_CART_CART_CART_CART_CART_CART_CART_CAR...,0_0_22967659_82448_432_10315_51_188_51_254173_...,"[36098226, 17729172, 34921664, 68317377, 69376...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-1, -1, -1, 68784030, 365470976025741936632...","[9227588, 52788392, 4585436, 87422901, 5736473...","[ORD, ORD, CART, CART, CART, CART, CART, CART,...","[0.0, 0.0, 22967659.0, 82448.0, 432.0, 10315.0..."
1,1254963427438884,30857980_44577431_1485384_63372823_19754701_29...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,9755613572694219_46916741_-1_-1_-1_-1_-1_4927...,18256630_46206270_65230659_86878921_32743602_5...,ORD_ORD_ORD_CART_CART_CART_CART_ORD_CART_CART_...,0_4813478_1115463_14061983_920_467229_304_1197...,"[30857980, 44577431, 1485384, 63372823, 197547...","[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[9755613572694219, 46916741, -1, -1, -1, -1, ...","[18256630, 46206270, 65230659, 86878921, 32743...","[ORD, ORD, ORD, CART, CART, CART, CART, ORD, C...","[0.0, 4813478.0, 1115463.0, 14061983.0, 920.0,..."
2,24374376067768939206305,42155506_90092899_4336287_74203839_39478459_37...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,-1_-1,8133925_59478082,CLICK_CLICK,0_27_49660,"[42155506, 90092899, 4336287, 74203839, 394784...","[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-1, -1]","[8133925, 59478082]","[CLICK, CLICK]","[0.0, 27.0, 49660.0]"
3,29463948632095349356845164880503931105432...,97234676_89827009_19973201_83859812_22172053_3...,1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,3426234151020854598336931355834712920960_3...,49854616_56947170_16552251_91485122_25087051_7...,CLICK_CLICK_CLICK_CLICK_CLICK_CLICK_CLICK_CLIC...,0_12_4_6_4_14_6_6_4_611_58095_1524_481_20_363_...,"[97234676, 89827009, 19973201, 83859812, 22172...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[3426234151020854598336931355834712920960,...","[49854616, 56947170, 16552251, 91485122, 25087...","[CLICK, CLICK, CLICK, CLICK, CLICK, CLICK, CLI...","[0.0, 12.0, 4.0, 6.0, 4.0, 14.0, 6.0, 6.0, 4.0..."
4,2793914964365783663289,19711065_83045333_8307571_20337381_11912759_53...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,25499566_24358647_606776896025741971768678_3...,68701_66196939_45800470_18419519_10090448_8073...,ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_OR...,0_0_2015453_958482_0_79443_854291_13678_119386...,"[19711065, 83045333, 8307571, 20337381, 119127...","[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[25499566, 24358647, 60677689602574197176867...","[68701, 66196939, 45800470, 18419519, 10090448...","[ORD, ORD, ORD, ORD, ORD, ORD, ORD, ORD, ORD, ...","[0.0, 0.0, 2015453.0, 958482.0, 0.0, 79443.0, ..."


### Numeric Features

In [6]:
# ============================================================
# FEATURE 1: candidate_count
# Number of candidates returned for the current query
# ============================================================

user_df["candidate_count"] = user_df["candidate_wid_list_parsed"].apply(
    lambda x: len(x) if isinstance(x, list) else np.nan
)

print("candidate_count")
print(f"Missing/unparseable lists: {user_df['candidate_count'].isna().sum():,}")
print(f"Zero candidate lists:      {(user_df['candidate_count'] == 0).sum():,}")
print(f"Negative values:           {(user_df['candidate_count'] < 0).sum():,}")

# ============================================================
# FEATURE 2: history_unique_query_count
# Number of distinct valid historical queries
# Excludes -1 (non-search interactions)
# ============================================================

def count_unique_queries(x):
    if not isinstance(x, list):
        return np.nan

    valid_queries = [q for q in x if q != "-1"]

    return len(set(valid_queries))


user_df["history_unique_query_count"] = (
    user_df["history_qry_list_parsed"].apply(count_unique_queries)
)

print("history_unique_query_count")
print(f"Missing/unparseable lists: {user_df['history_unique_query_count'].isna().sum():,}")
print(f"Zero unique queries:       {(user_df['history_unique_query_count'] == 0).sum():,}")
print(f"Negative values:           {(user_df['history_unique_query_count'] < 0).sum():,}")

# ============================================================
# FEATURE 3: history_unique_wid_count
# Number of distinct products interacted with historically
# ============================================================

user_df["history_unique_wid_count"] = user_df["history_wid_list_parsed"].apply(
    lambda x: len(set(x)) if isinstance(x, list) else np.nan
)

print("history_unique_wid_count")
print(f"Missing/unparseable lists: {user_df['history_unique_wid_count'].isna().sum():,}")
print(f"Zero unique WIDs:          {(user_df['history_unique_wid_count'] == 0).sum():,}")
print(f"Negative values:           {(user_df['history_unique_wid_count'] < 0).sum():,}")

candidate_count
Missing/unparseable lists: 0
Zero candidate lists:      0
Negative values:           0


history_unique_query_count
Missing/unparseable lists: 0
Zero unique queries:       4,430
Negative values:           0


history_unique_wid_count
Missing/unparseable lists: 0
Zero unique WIDs:          0
Negative values:           0


In [7]:
# ============================================================
# FEATURES 4-6: Candidate interaction rates
# IMPORTANT: Replace with actual JDSearch label mapping
# ============================================================

CLICK_LABEL = 1.0
CART_LABEL = 2.0
PURCHASE_LABEL = 3.0

def candidate_rate(labels, target_label):
    """
    Proportion of candidates receiving target_label.

    Returns NaN when:
    - parsed value is not a list
    - candidate label list is empty
    """
    if not isinstance(labels, list) or len(labels) == 0:
        return np.nan

    return sum(label == target_label for label in labels) / len(labels)


# Check denominator problems before calculating rates
invalid_label_lists = ~user_df["candidate_label_list_parsed"].apply(
    lambda x: isinstance(x, list)
)

empty_label_lists = user_df["candidate_label_list_parsed"].apply(
    lambda x: isinstance(x, list) and len(x) == 0
)

print("Candidate rate denominator checks")
print(f"Unparseable/non-list values: {invalid_label_lists.sum():,}")
print(f"Empty lists (zero denominator): {empty_label_lists.sum():,}")
print("These observations will receive NaN for candidate rates.")


# Calculate rates
user_df["candidate_click_rate"] = (
    user_df["candidate_label_list_parsed"]
    .apply(lambda x: candidate_rate(x, CLICK_LABEL))
)

user_df["candidate_cart_rate"] = (
    user_df["candidate_label_list_parsed"]
    .apply(lambda x: candidate_rate(x, CART_LABEL))
)

user_df["candidate_purchase_rate"] = (
    user_df["candidate_label_list_parsed"]
    .apply(lambda x: candidate_rate(x, PURCHASE_LABEL))
)


# Validate resulting rates
for col in [
    "candidate_click_rate",
    "candidate_cart_rate",
    "candidate_purchase_rate"
]:
    print(f"\n{col}")
    print(f"Missing:       {user_df[col].isna().sum():,}")
    print(f"Below 0:       {(user_df[col] < 0).sum():,}")
    print(f"Above 1:       {(user_df[col] > 1).sum():,}")

Candidate rate denominator checks
Unparseable/non-list values: 0
Empty lists (zero denominator): 0
These observations will receive NaN for candidate rates.



candidate_click_rate
Missing:       0
Below 0:       0
Above 1:       0

candidate_cart_rate
Missing:       0
Below 0:       0
Above 1:       0

candidate_purchase_rate
Missing:       0
Below 0:       0
Above 1:       0


In [8]:
# ============================================================
# FEATURES 7-10: Historical action counts
# CLICK, CART, ORD, FLW
# ============================================================

ACTION_MAP = {
    "history_click_count": "CLICK",
    "history_cart_count": "CART",
    "history_order_count": "ORD",
    "history_follow_count": "FLW"
}

for col, action in ACTION_MAP.items():

    user_df[col] = user_df["history_type_list_parsed"].apply(
        lambda x: x.count(action) if isinstance(x, list) else np.nan
    )

    print(f"\n{col}")
    print(f"Missing/unparseable lists: {user_df[col].isna().sum():,}")
    print(f"Zero counts:               {(user_df[col] == 0).sum():,}")
    print(f"Negative values:           {(user_df[col] < 0).sum():,}")


history_click_count
Missing/unparseable lists: 0
Zero counts:               28,184
Negative values:           0

history_cart_count
Missing/unparseable lists: 0
Zero counts:               7,449
Negative values:           0



history_order_count
Missing/unparseable lists: 0
Zero counts:               14,686
Negative values:           0

history_follow_count
Missing/unparseable lists: 0
Zero counts:               171,692
Negative values:           0


In [9]:
# ============================================================
# FEATURE 11: history_length
# Number of historically interacted products
# ============================================================

user_df["history_length"] = user_df["history_wid_list_parsed"].apply(
    lambda x: len(x) if isinstance(x, list) else np.nan
)

print("history_length")
print(f"Missing/unparseable lists: {user_df['history_length'].isna().sum():,}")
print(f"Zero-length histories:     {(user_df['history_length'] == 0).sum():,}")
print(f"Negative values:           {(user_df['history_length'] < 0).sum():,}")

# ============================================================
# FEATURE 12: candidate_seen_history_count
# Number of current candidates previously interacted with
# ============================================================

def count_seen_candidates(row):
    candidates = row["candidate_wid_list_parsed"]
    history = row["history_wid_list_parsed"]

    if not isinstance(candidates, list) or not isinstance(history, list):
        return np.nan

    return len(set(candidates) & set(history))


user_df["candidate_seen_history_count"] = user_df.apply(
    count_seen_candidates,
    axis=1
)

print("candidate_seen_history_count")
print(f"Missing/unparseable lists: {user_df['candidate_seen_history_count'].isna().sum():,}")
print(f"Zero overlap:              {(user_df['candidate_seen_history_count'] == 0).sum():,}")
print(f"Negative values:           {(user_df['candidate_seen_history_count'] < 0).sum():,}")

# Logical consistency check
invalid_overlap = (
    user_df["candidate_seen_history_count"] > user_df["candidate_count"]
)

print(
    f"Count exceeding candidate_count: "
    f"{invalid_overlap.sum():,}"
)

history_length
Missing/unparseable lists: 0
Zero-length histories:     0
Negative values:           0


candidate_seen_history_count
Missing/unparseable lists: 0
Zero overlap:              123,808
Negative values:           0
Count exceeding candidate_count: 0


In [10]:
# ============================================================
# FEATURE 13: time_since_last_action
# ============================================================

def get_last_interval(time_list):
    if len(time_list) > 0:
        return float(time_list[-1])
    return np.nan

user_df["time_since_last_action"] = (
    user_df["history_time_list_parsed"].apply(get_last_interval)
)

# Data quality checks
print("time_since_last_action")
print(f"Missing values:   {user_df['time_since_last_action'].isna().sum():,}")
print(f"Zero values:      {(user_df['time_since_last_action'] == 0).sum():,}")
print(f"Negative values:  {(user_df['time_since_last_action'] < 0).sum():,}")

time_since_last_action
Missing values:   0
Zero values:      0
Negative values:  0


In [11]:
# ============================================================
# NUMERIC FEATURE SUMMARY
# ============================================================

numeric_features = [
    "candidate_count",
    "history_unique_query_count",
    "history_unique_wid_count",
    "candidate_click_rate",
    "candidate_cart_rate",
    "candidate_purchase_rate",
    "history_click_count",
    "history_cart_count",
    "history_order_count",
    "history_follow_count",
    "history_length",
    "candidate_seen_history_count",
    "time_since_last_action"
]


numeric_summary = pd.DataFrame({
    "Min": user_df[numeric_features].min(),
    "Max": user_df[numeric_features].max(),
    "Mean": user_df[numeric_features].mean(),
    "Median": user_df[numeric_features].median(),
    "SD": user_df[numeric_features].std(),
    "Missing": user_df[numeric_features].isna().sum()
})
numeric_summary = numeric_summary.round(5)

numeric_summary

,Min,Max,Mean,Median,SD,Missing
candidate_count,1.0,200.0,89.22466,90.00000,3.885736e+01,0
history_unique_query_count,0.0,1330.0,29.71442,17.00000,3.888620e+01,0
history_unique_wid_count,1.0,3647.0,99.64908,52.00000,1.398403e+02,0
candidate_click_rate,0.0,1.0,0.02060,0.01111,5.587000e-02,0
candidate_cart_rate,0.0,1.0,0.00803,0.00000,4.016000e-02,0
candidate_purchase_rate,0.0,1.0,0.00200,0.00000,1.781000e-02,0
history_click_count,0.0,3575.0,47.74116,17.00000,9.333230e+01,0
history_cart_count,0.0,3614.0,66.30868,29.00000,1.150793e+02,0
history_order_count,0.0,2962.0,39.21788,15.00000,7.040689e+01,0
history_follow_count,0.0,695.0,0.14140,0.00000,4.006920e+00,0


In [12]:
# ============================================================
# VALIDATION: Historical interaction counts
# ============================================================
# A. Candidate Label Distribution (Converted to int for cleaner display)
all_labels = [int(label) for sublist in user_df['candidate_label_list_parsed'] for label in sublist]
print("Candidate Label Dist (0:None, 1:Click, 2:Cart, 3:Purchase):")
print(pd.Series(all_labels).value_counts().sort_index())

# B. History Type Distribution
all_types = [t for sublist in user_df['history_type_list_parsed'] for t in sublist]
print("\nHistory Interaction Type Distribution:")
print(pd.Series(all_types).value_counts())


print("Historical CLICK")
print("Flattened count:",
      (pd.Series(all_types) == "CLICK").sum())
print("Sum of engineered counts:",
      user_df["history_click_count"].sum())

print("\nHistorical CART")
print("Flattened count:",
      (pd.Series(all_types) == "CART").sum())
print("Sum of engineered counts:",
      user_df["history_cart_count"].sum())

print("\nHistorical ORD")
print("Flattened count:",
      (pd.Series(all_types) == "ORD").sum())
print("Sum of engineered counts:",
      user_df["history_order_count"].sum())

print("\nHistorical FOLLOW")
print("Flattened count:",
      (pd.Series(all_types) == "FLW").sum())
print("Sum of engineered counts:",
      user_df["history_follow_count"].sum())

# ============================================================
# VALIDATION: Candidate interaction counts
# ============================================================

labels = pd.Series(all_labels)

print("Candidate CLICK")
print("Flattened count:",
      (labels == 1).sum())
print("Recovered from engineered features:",
      (user_df["candidate_click_rate"] *
       user_df["candidate_count"]).sum())

print("\nCandidate CART")
print("Flattened count:",
      (labels == 2).sum())
print("Recovered from engineered features:",
      (user_df["candidate_cart_rate"] *
       user_df["candidate_count"]).sum())

print("\nCandidate PURCHASE")
print("Flattened count:",
      (labels == 3).sum())
print("Recovered from engineered features:",
      (user_df["candidate_purchase_rate"] *
       user_df["candidate_count"]).sum())

Candidate Label Dist (0:None, 1:Click, 2:Cart, 3:Purchase):


0    15189116
1      224487
2       75573
3       20836
Name: count, dtype: int64



History Interaction Type Distribution:


CART     11526504
CLICK     8298894
ORD       6817283
FLW         24579
Name: count, dtype: int64
Historical CLICK


Flattened count: 8298894
Sum of engineered counts: 8298894

Historical CART


Flattened count: 11526504
Sum of engineered counts: 11526504

Historical ORD


Flattened count: 6817283
Sum of engineered counts: 6817283

Historical FOLLOW


Flattened count: 24579
Sum of engineered counts: 24579


Candidate CLICK
Flattened count: 224487
Recovered from engineered features: 224487.0

Candidate CART
Flattened count: 75573
Recovered from engineered features: 75573.0

Candidate PURCHASE
Flattened count: 20836
Recovered from engineered features: 20836.0


## Categorical Features

In [13]:
# ============================================================
# CATEGORICAL / ID FEATURE SUMMARY
# ============================================================

categorical_features = [
    "candidate_wid_list_parsed",
    "candidate_label_list_parsed",
    "history_qry_list_parsed",
    "history_wid_list_parsed",
    "history_type_list_parsed"
]

categorical_summary = []

for col in categorical_features:

    # Missing lists at row level
    missing_count = user_df[col].isna().sum()

    # Flatten all lists
    values = user_df[col].dropna().explode()

    # Remove missing values within lists
    values = values.dropna()

    # Distinct categories
    distinct_count = values.nunique()

    # Representative values
    representative_values = ", ".join(
        map(str, values.value_counts().head(5).index)
    )

    categorical_summary.append({
        "Feature": col,
        "Distinct Categories": distinct_count,
        "Representative Values": representative_values,
        "Missing": missing_count
    })

# ============================================================
# CANDIDATE LABEL DISTRIBUTION
# 0 = None, 1 = Click, 2 = Cart, 3 = Purchase
# ============================================================

all_labels = [
    int(label)
    for sublist in user_df["candidate_label_list_parsed"]
    for label in sublist
]

candidate_label_dist = pd.Series(all_labels).value_counts().sort_index()

candidate_label_summary = pd.DataFrame({
    "Count": candidate_label_dist,
    "Percent": candidate_label_dist / candidate_label_dist.sum() * 100
})

candidate_label_summary["Percent"] = candidate_label_summary["Percent"].round(5)

print("Candidate Label Distribution:")
display(candidate_label_summary)

# ============================================================
# HISTORY INTERACTION TYPE DISTRIBUTION
# ============================================================

all_types = [
    interaction
    for sublist in user_df["history_type_list_parsed"]
    for interaction in sublist
]

history_type_dist = pd.Series(all_types).value_counts()

history_type_summary = pd.DataFrame({
    "Count": history_type_dist,
    "Percent": history_type_dist / history_type_dist.sum() * 100
})

history_type_summary["Percent"] = history_type_summary["Percent"].round(5)

print("History Interaction Type Distribution:")
display(history_type_summary)


categorical_summary = pd.DataFrame(categorical_summary)

categorical_summary

Candidate Label Distribution:


,Count,Percent
0,15189116,97.93104
1,224487,1.44737
2,75573,0.48725
3,20836,0.13434


History Interaction Type Distribution:


,Count,Percent
CART,11526504,43.22343
CLICK,8298894,31.12016
ORD,6817283,25.56424
FLW,24579,0.09217


,Feature,Distinct Categories,Representative Values,Missing
0,candidate_wid_list_parsed,8362535,"26194387, 64338053, 71524113, 99727072, 81138348",0
1,candidate_label_list_parsed,4,"0.0, 1.0, 2.0, 3.0",0
2,history_qry_list_parsed,1691283,"-1, 61485993, 59054959, 88843206, 94032448",0
3,history_wid_list_parsed,5584543,"97745171, 99727072, 81138348, 24157445, 76456085",0
4,history_type_list_parsed,4,"CART, CLICK, ORD, FLW",0


In [ ]:
import gc

# Free the parsed lists from the user section before loading the 12M-row product catalogue
for name in ["all_labels", "all_types", "labels", "values"]:
    globals().pop(name, None)
user_df = user_df.drop(columns=[c for c in user_df.columns if c.endswith("_parsed")])
gc.collect()

## Product Metadata: Descriptive Statistics

Each row of `product_meta_data.txt` describes one product. All IDs and term ids are anonymised.

In [14]:
product_raw_dictionary = pd.DataFrame(
    [
        ("wid", "Product id (primary key)", "categorical ID"),
        ("name", "Product name as a sequence of term ids", "text (term-id sequence)"),
        ("brand_id", "Brand id", "categorical ID"),
        ("brand_name", "Brand name as a sequence of term ids", "text (term-id sequence)"),
    ]
    # Four category levels, from level 1 (broadest) to level 4 (leaf)
    + [
        row
        for level in range(1, 5)
        for row in [
            (f"cate_id_{level}", f"Level-{level} category id", "categorical ID"),
            (f"cate_name_{level}", f"Level-{level} category name as a sequence of term ids", "text (term-id sequence)"),
        ]
    ]
    + [("shop_id", "Shop id (-1 = no shop)", "categorical ID")],
    columns=["Column", "Description", "Type"],
).set_index("Column")
product_raw_dictionary

,Description,Type
Column,,
wid,Product id (primary key),categorical ID
name,Product name as a sequence of term ids,text (term-id sequence)
brand_id,Brand id,categorical ID
brand_name,Brand name as a sequence of term ids,text (term-id sequence)
cate_id_1,Level-1 category id,categorical ID
cate_name_1,Level-1 category name as a sequence of term ids,text (term-id sequence)
cate_id_2,Level-2 category id,categorical ID
cate_name_2,Level-2 category name as a sequence of term ids,text (term-id sequence)
cate_id_3,Level-3 category id,categorical ID


### Reading the File in Chunks

The file holds 12.1M products, too many to load as strings in one go, so we read it in chunks and keep running statistics for each column.

In [15]:
import time

# Term ids inside name / brand_name / cate_name_* are separated by \x18
TERM_SEP = "\x18"

PRODUCT_COLS = [
    "wid", "name", "brand_id", "brand_name",
    "cate_id_1", "cate_name_1", "cate_id_2", "cate_name_2",
    "cate_id_3", "cate_name_3", "cate_id_4", "cate_name_4",
    "shop_id",
]
ID_COLS = ["brand_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4", "shop_id"]
TEXT_COLS = ["name", "brand_name", "cate_name_1", "cate_name_2", "cate_name_3", "cate_name_4"]
CHUNK_SIZE = 200_000


class OnlineStats:
    """Running mean, std, min and max without storing every value (Welford's algorithm)."""

    def __init__(self):
        self.n, self.mean, self.M2 = 0, 0.0, 0.0
        self.min, self.max = None, None

    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        self.M2 += delta * (x - self.mean)
        self.min = x if self.min is None else min(self.min, x)
        self.max = x if self.max is None else max(self.max, x)

    @property
    def std(self):
        return (self.M2 / self.n) ** 0.5 if self.n > 1 else 0.0


def parse_term_list(val):
    """Split a term-id field into its list of term ids."""
    if pd.isna(val) or val == "":
        return []
    return val.split(TERM_SEP)


# Running totals, updated one chunk at a time
row_count = 0
missing_counts = {c: 0 for c in PRODUCT_COLS}
distinct_ids = {c: set() for c in ID_COLS}
id_top_values = {c: Counter() for c in ID_COLS}
text_len_stats = {c: OnlineStats() for c in TEXT_COLS}
text_vocab = {c: set() for c in TEXT_COLS}

start = time.time()
for chunk in pd.read_csv(PRODUCT_TXT, sep="\t", names=PRODUCT_COLS, header=0, dtype=str, chunksize=CHUNK_SIZE):
    row_count += len(chunk)

    # Missing values per column
    for c in PRODUCT_COLS:
        missing_counts[c] += int(chunk[c].isna().sum())

    # Distinct and most frequent values of each ID column
    for c in ID_COLS:
        values = chunk[c].dropna()
        distinct_ids[c].update(values.unique())
        id_top_values[c].update(values)

    # Length (in term ids) and vocabulary of each text field
    for c in TEXT_COLS:
        for terms in chunk[c].dropna().apply(parse_term_list):
            text_len_stats[c].update(len(terms))
            text_vocab[c].update(terms)

print(f"Processed {row_count:,} products in {time.time() - start:.0f}s")

Processed 12,141,247 products in 133s


### ID Columns

Brand, shop and category IDs are categorical. For each we report the number of distinct values, missing values and the five most frequent values.

In [16]:
# Distinct values, missing values and top-5 values of each ID column
id_summary = pd.DataFrame({
    "Distinct Categories": {c: len(distinct_ids[c]) for c in ID_COLS},
    "Missing": {c: missing_counts[c] for c in ID_COLS},
    "Representative Values": {c: ", ".join(v for v, _ in id_top_values[c].most_common(5)) for c in ID_COLS},
})

# wid is one per product (no duplicates, verified in the coverage check below)
id_summary.loc["wid"] = [row_count, missing_counts["wid"], "one per product"]
id_summary = id_summary.loc[["wid"] + ID_COLS]
id_summary

,Distinct Categories,Missing,Representative Values
wid,12141247,0,one per product
brand_id,182585,0,"38645246, 54289642, 33241674, 67072585, 98209701"
cate_id_1,76,0,"68949654, 24093831, 920973, 98371044, 31606173"
cate_id_2,671,0,"92351230, 18907346, 33276684, 71940523, 38845680"
cate_id_3,5819,0,"72526272, 10287851, 64184092, 21386661, 94549943"
cate_id_4,7137,0,"32130995, 462318, 23794926, 97122609, 76679610"
shop_id,250935,0,"-1, 21749031, 54486560, 58139116, 74969976"


### Text Columns

Product names, brand names and category names are sequences of anonymised term ids. For each we report the length in terms and the vocabulary size.

In [17]:
# Length (in term ids) and vocabulary size of each text field
text_summary = pd.DataFrame({
    "Mean Length": {c: round(text_len_stats[c].mean, 3) for c in TEXT_COLS},
    "Std Length": {c: round(text_len_stats[c].std, 3) for c in TEXT_COLS},
    "Min Length": {c: text_len_stats[c].min for c in TEXT_COLS},
    "Max Length": {c: text_len_stats[c].max for c in TEXT_COLS},
    "Vocab Size (distinct term ids)": {c: len(text_vocab[c]) for c in TEXT_COLS},
    "Missing": {c: missing_counts[c] for c in TEXT_COLS},
})
text_summary

,Mean Length,Std Length,Min Length,Max Length,Vocab Size (distinct term ids),Missing
name,27.913,8.056,1,78,986404,0
brand_name,2.561,1.731,1,31,192684,941660
cate_name_1,1.774,0.663,1,9,128,0
cate_name_2,1.917,0.690,1,7,751,0
cate_name_3,1.913,0.938,1,15,4996,0
cate_name_4,1.945,0.951,1,15,5926,0


### Metadata Coverage & Data Checks

Only products that appear in the user logs matter for modelling, so we check how many logged candidates and history products have metadata. We also check three things raised by the tables above:
- whether level-4 categories just repeat level 3
- how common the `shop_id = -1` placeholder is
- which `brand_id`s the products without a `brand_name` belong to

In [18]:
def wid_occurrences(series):
    """Count how often each product id appears across '_'-separated id lists."""
    counts = Counter()
    for val in series.dropna():
        counts.update(x for x in val.split("_") if x)
    return counts


# How often each product appears as a candidate and in user histories
candidate_wids = wid_occurrences(user_df["candidate_wid_list"])
history_wids = wid_occurrences(user_df["history_wid_list"])

# Scan the catalogue for its product ids and the category / shop / brand checks
catalog_wids = set()
cate_pairs = set()
brand_rows, no_name_brand_rows = Counter(), Counter()
duplicate_wids = same_cate_id_34 = same_cate_name_34 = shop_missing = 0

check_cols = ["wid", "brand_id", "brand_name", "cate_id_3", "cate_name_3", "cate_id_4", "cate_name_4", "shop_id"]
for chunk in pd.read_csv(PRODUCT_TXT, sep="\t", names=PRODUCT_COLS, header=0, dtype=str,
                         usecols=check_cols, chunksize=500_000):
    before = len(catalog_wids)
    catalog_wids.update(chunk["wid"])
    duplicate_wids += len(chunk) - (len(catalog_wids) - before)

    same_cate_id_34 += int((chunk["cate_id_3"] == chunk["cate_id_4"]).sum())
    same_cate_name_34 += int((chunk["cate_name_3"] == chunk["cate_name_4"]).sum())
    cate_pairs.update(zip(chunk["cate_id_3"], chunk["cate_id_4"]))

    shop_missing += int((chunk["shop_id"] == "-1").sum())
    brand_rows.update(chunk["brand_id"])
    no_name_brand_rows.update(chunk.loc[chunk["brand_name"].isna(), "brand_id"])

# Share of distinct products and of all appearances that have metadata
coverage_rows = []
for source, counts in [("candidate", candidate_wids), ("history", history_wids)]:
    distinct_covered = sum(1 for w in counts if w in catalog_wids)
    total_occurrences = sum(counts.values())
    covered_occurrences = sum(n for w, n in counts.items() if w in catalog_wids)
    coverage_rows.append({
        "Source": source,
        "Distinct Products": len(counts),
        "Distinct With Metadata": distinct_covered,
        "Distinct Coverage %": round(100 * distinct_covered / len(counts), 3),
        "Occurrences": total_occurrences,
        "Occurrences With Metadata": covered_occurrences,
        "Occurrence Coverage %": round(100 * covered_occurrences / total_occurrences, 3),
    })

logged_wids = set(candidate_wids) | set(history_wids)
print(f"Distinct products in logs (candidate or history): {len(logged_wids):,}")
print(f"Products in both candidate lists and histories:   {len(set(candidate_wids) & set(history_wids)):,}")
print(f"Catalogue products never seen in the logs:        {len(catalog_wids - logged_wids):,}")
print(f"Duplicate wids in catalogue:                      {duplicate_wids:,}")

# Level 4 vs level 3: same id, same name, and how many level-4 children each level-3 category has
children_per_cate3 = pd.Series([c3 for c3, _ in cate_pairs]).value_counts()
print(f"\ncate_id_4 == cate_id_3:     {same_cate_id_34:,} ({same_cate_id_34 / row_count:.2%})")
print(f"cate_name_4 == cate_name_3: {same_cate_name_34:,} ({same_cate_name_34 / row_count:.2%})")
print(f"Level-3 categories with a single level-4 child: {(children_per_cate3 == 1).mean():.1%}")

# Placeholders: shop -1, and the brand_ids holding products without a brand_name
print(f"\nshop_id == -1: {shop_missing:,} ({shop_missing / row_count:.2%})")
print(f"Products without brand_name: {sum(no_name_brand_rows.values()):,}, across {len(no_name_brand_rows)} brand_ids:")
for brand, n in no_name_brand_rows.most_common():
    print(f"  brand_id {brand}: {n:,} of its {brand_rows[brand]:,} products have no brand_name")

coverage_summary = pd.DataFrame(coverage_rows).set_index("Source")

# Free memory before building features
del candidate_wids, history_wids, catalog_wids, logged_wids

coverage_summary

Distinct products in logs (candidate or history): 12,319,978


Products in both candidate lists and histories:   1,627,100
Catalogue products never seen in the logs:        552,658
Duplicate wids in catalogue:                      0

cate_id_4 == cate_id_3:     0 (0.00%)
cate_name_4 == cate_name_3: 10,353,605 (85.28%)
Level-3 categories with a single level-4 child: 95.1%

shop_id == -1: 103,162 (0.85%)
Products without brand_name: 941,660, across 3 brand_ids:
  brand_id 38645246: 519,781 of its 519,781 products have no brand_name
  brand_id 54289642: 336,110 of its 336,110 products have no brand_name
  brand_id 98209701: 85,769 of its 88,444 products have no brand_name


,Distinct Products,Distinct With Metadata,Distinct Coverage %,Occurrences,Occurrences With Metadata,Occurrence Coverage %
Source,,,,,,
candidate,8362535,8305622,99.319,15510012,15439674,99.546
history,5584543,4906914,87.866,26667260,25371172,95.140


### Findings: Metadata Coverage & Data Quality

- **Candidates are almost fully covered.** 99.32% of distinct candidate products (99.55% of candidate slots) have metadata, so product features are available for nearly every candidate we rank.
- **History is mostly covered.** 87.87% of distinct history products (95.14% of history interactions) have metadata. The remaining 4.86% of interactions should be kept and mapped to an "unknown product" placeholder rather than dropped. The previous group's pipeline deleted history items without metadata, and because it had also reduced the catalogue to sampled candidates only, mean history length fell from 153 to 66 interactions.
- **The catalogue is clean.** 12,141,247 products with no duplicate `wid`. Only 552,658 (4.55%) never appear in the logs, so the full-catalogue statistics above closely reflect the products we model.
- **Category level 4 mostly repeats level 3.** `cate_id_4` never equals `cate_id_3`, because IDs are numbered separately per level, but `cate_name_4` equals `cate_name_3` for 85.28% of products, and 95.1% of level-3 categories have exactly one level-4 child. Most products are therefore only 3 levels deep, and a separate level-4 embedding (as the previous group used) mostly duplicates level 3. This is captured below as `cate_depth`.
- **Brand and shop contain placeholders.** `shop_id = -1` (103,162 products, 0.85%) means "no shop". All 941,660 products without a `brand_name` fall into just 3 `brand_id`s, two of which contain nothing else. These act as "no brand" buckets, so the largest "brand" (519,781 products) is a placeholder, not a real brand. Brand- and shop-level features below exclude these placeholders.

## Product-Level Features

We build one row of features per catalogue product and save it to `JDsearch/derived/product_features.parquet`, to be joined onto each query's candidates later.

Popularity features are built from users' interaction histories only, never from `candidate_label_list`, because that column is the prediction target. The previous group measured category popularity by catalogue size (supply), which says nothing about what users actually engage with.

### a. Product Attributes & Supply-Side Size

- `name_length`: number of term ids in the product name
- `has_brand`: whether the product has a brand name
- `shop_is_missing`: whether `shop_id` is the `-1` placeholder
- `cate_depth`: 3 if the level-4 category repeats level 3, otherwise 4
- `brand_product_count`, `shop_product_count`, `cate4_product_count`: how many catalogue products share the product's brand, shop or leaf category (`cate_id_4`). Placeholder brands and shops get no value.

In [19]:
ID_COLS_INT = ["wid", "brand_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4", "shop_id"]
TEXT_CHECK_COLS = ["name", "brand_name", "cate_name_3", "cate_name_4"]

start = time.time()
parts = []
for chunk in pd.read_csv(PRODUCT_TXT, sep="\t", names=PRODUCT_COLS, header=0, dtype=str,
                         usecols=ID_COLS_INT + TEXT_CHECK_COLS, chunksize=1_000_000):
    # Store IDs as integers so the full catalogue fits in memory
    part = chunk[ID_COLS_INT].astype("int64")

    # Attributes read directly from the product row
    part["name_length"] = (chunk["name"].str.count(TERM_SEP) + 1).astype("int16")
    part["has_brand"] = chunk["brand_name"].notna().astype("int8")
    part["shop_is_missing"] = (part["shop_id"] == -1).astype("int8")
    part["cate_depth"] = np.where(chunk["cate_name_3"] == chunk["cate_name_4"], 3, 4).astype("int8")
    parts.append(part)

product_df = pd.concat(parts, ignore_index=True)
del parts

# Catalogue size of each brand / shop / leaf category, skipping placeholder brands and shop -1
real_brand = product_df["has_brand"] == 1
real_shop = product_df["shop_id"] != -1
product_df["brand_product_count"] = product_df[real_brand].groupby("brand_id")["wid"].transform("size").astype("float32")
product_df["shop_product_count"] = product_df[real_shop].groupby("shop_id")["wid"].transform("size").astype("float32")
product_df["cate4_product_count"] = product_df.groupby("cate_id_4")["wid"].transform("size").astype("int32")

print(f"Built attributes for {len(product_df):,} products in {time.time() - start:.0f}s")
product_df.head()

Built attributes for 12,141,247 products in 35s


,wid,brand_id,cate_id_1,cate_id_2,cate_id_3,cate_id_4,shop_id,name_length,has_brand,shop_is_missing,cate_depth,brand_product_count,shop_product_count,cate4_product_count
0,84303384,81493954,22259905,80083037,13599099,50485737,86521100,13,1,0,4,80.0,9145.0,8069
1,58965086,15444334,68949654,58682510,37322654,99504987,1462363,29,1,0,3,1258.0,771.0,22353
2,33811859,39479430,22259905,80083037,93252909,42385237,86521100,19,1,0,4,6.0,9145.0,9193
3,90290292,75019956,75739187,9682321,56674307,65420542,9644527,31,1,0,3,8225.0,157.0,12796
4,28171378,55412149,44687275,38428440,56166561,60897751,60136421,34,1,0,3,14.0,13.0,2245


### b. Demand-Side Popularity

How often each product was clicked, added to cart, purchased or followed across all users' histories:
- `hist_click_count`, `hist_cart_count`, `hist_ord_count`, `hist_flw_count`
- `hist_total_count`: all history interactions with the product

In [20]:
ACTION_TYPES = ["CLICK", "CART", "ORD", "FLW"]
USER_BATCH = 20_000

start = time.time()
partial_counts = []
for lo in range(0, len(user_df), USER_BATCH):
    batch = user_df.iloc[lo:lo + USER_BATCH]

    # One row per history interaction: (product id, interaction type)
    wids = batch["history_wid_list"].str.split("_").explode()
    types = batch["history_type_list"].str.split("_").explode()
    assert len(wids) == len(types), "history_wid_list and history_type_list lengths differ"
    pairs = pd.DataFrame({"wid": wids.to_numpy().astype("int64"), "type": types.to_numpy()})

    # Interaction counts per product and type for this batch of users
    partial_counts.append(pairs.groupby(["wid", "type"]).size())

# Combine batches: one row per product, one column per interaction type
history_counts = (
    pd.concat(partial_counts).groupby(level=["wid", "type"]).sum()
    .unstack("type", fill_value=0)
    .reindex(columns=ACTION_TYPES, fill_value=0)
)
del partial_counts
history_counts.columns = [f"hist_{t.lower()}_count" for t in ACTION_TYPES]
history_counts["hist_total_count"] = history_counts.sum(axis=1)

print(f"{len(history_counts):,} products have history interactions "
      f"({int(history_counts['hist_total_count'].sum()):,} interactions in total), built in {time.time() - start:.0f}s")
print(f"{(~history_counts.index.isin(product_df['wid'])).sum():,} of these products are missing from the catalogue")
history_counts.head()

5,584,543 products have history interactions (26,667,260 interactions in total), built in 10s


677,629 of these products are missing from the catalogue


,hist_click_count,hist_cart_count,hist_ord_count,hist_flw_count,hist_total_count
wid,,,,,
2,0,1,0,0,1
11,0,0,1,0,1
39,0,1,0,0,1
42,0,0,1,0,1
67,1,0,0,0,1


### c. Brand & Leaf-Category Popularity

Most products have few or no interactions, so popularity is also rolled up to the brand and the leaf category. Purchase rates (`ORD` / all interactions) are smoothed towards the global rate by adding 20 pseudo-interactions, so a product bought once in one interaction doesn't get a rate of 100%.
- `prod_order_rate`, `brand_order_rate`, `cate4_order_rate`: smoothed purchase rates
- `brand_hist_total_count`, `cate4_hist_total_count`: all history interactions with the brand's or leaf category's products

In [21]:
COUNT_COLS = list(history_counts.columns)
SMOOTHING = 20
global_order_rate = history_counts["hist_ord_count"].sum() / history_counts["hist_total_count"].sum()


def smoothed_order_rate(orders, total):
    """Purchase rate pulled towards the global rate by SMOOTHING pseudo-interactions."""
    return ((orders + SMOOTHING * global_order_rate) / (total + SMOOTHING)).astype("float32")


# Attach history counts to the catalogue; products never interacted with get 0
product_df = product_df.merge(history_counts, how="left", left_on="wid", right_index=True)
product_df[COUNT_COLS] = product_df[COUNT_COLS].fillna(0).astype("int32")

# Product-level purchase rate
product_df["prod_order_rate"] = smoothed_order_rate(product_df["hist_ord_count"], product_df["hist_total_count"])

# Roll counts up to brand (real brands only) and leaf category
for level, key, rows in [("brand", "brand_id", product_df["has_brand"] == 1), ("cate4", "cate_id_4", slice(None))]:
    totals = product_df.loc[rows].groupby(key)[["hist_total_count", "hist_ord_count"]].transform("sum")
    product_df[f"{level}_hist_total_count"] = totals["hist_total_count"].astype("float32")
    product_df[f"{level}_order_rate"] = smoothed_order_rate(totals["hist_ord_count"], totals["hist_total_count"])

print(f"Global purchase rate: {global_order_rate:.4f}")
print(f"Products with no history interactions: {(product_df['hist_total_count'] == 0).mean():.2%}")

# Save for joining onto query candidates later
DERIVED_DIR = BASE / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
product_df.to_parquet(DERIVED_DIR / "product_features.parquet", index=False)

Global purchase rate: 0.2556
Products with no history interactions: 59.58%


### Feature Dictionary

Description, level, type and descriptive statistics of every product-level feature.

In [22]:
product_feature_dictionary = pd.DataFrame([
    ("name_length", "Product", "Number of term ids in the product name", "count (terms)"),
    ("has_brand", "Product", "1 if brand_name is present, else 0", "binary"),
    ("shop_is_missing", "Product", "1 if shop_id is the -1 placeholder, else 0", "binary"),
    ("cate_depth", "Product", "3 if the level-4 category repeats level 3 (same name), else 4", "ordinal (3/4)"),
    ("brand_product_count", "Brand", "Catalogue products under the same brand (missing if no brand_name)", "count (products)"),
    ("shop_product_count", "Shop", "Catalogue products in the same shop (missing for shop_id = -1)", "count (products)"),
    ("cate4_product_count", "Leaf category", "Catalogue products in the same leaf category", "count (products)"),
    ("hist_click_count", "Product", "Clicks on the product across all users' histories", "count (interactions)"),
    ("hist_cart_count", "Product", "Add-to-carts across all users' histories", "count (interactions)"),
    ("hist_ord_count", "Product", "Purchases across all users' histories", "count (interactions)"),
    ("hist_flw_count", "Product", "Follows across all users' histories", "count (interactions)"),
    ("hist_total_count", "Product", "All history interactions with the product", "count (interactions)"),
    ("prod_order_rate", "Product", f"Purchases / interactions, smoothed with {SMOOTHING} pseudo-interactions", "rate (0-1)"),
    ("brand_hist_total_count", "Brand", "All history interactions with the brand's products (missing if no brand_name)", "count (interactions)"),
    ("brand_order_rate", "Brand", "Smoothed purchase rate over the brand's products (missing if no brand_name)", "rate (0-1)"),
    ("cate4_hist_total_count", "Leaf category", "All history interactions within the leaf category", "count (interactions)"),
    ("cate4_order_rate", "Leaf category", "Smoothed purchase rate within the leaf category", "rate (0-1)"),
], columns=["Feature", "Level", "Description", "Type / Unit"]).set_index("Feature")

# Descriptive statistics of each feature across all catalogue products
feature_cols = list(product_feature_dictionary.index)
product_feature_stats = pd.DataFrame({
    "Min": product_df[feature_cols].min(),
    "Max": product_df[feature_cols].max(),
    "Mean": product_df[feature_cols].mean(),
    "Median": product_df[feature_cols].median(),
    "SD": product_df[feature_cols].std(),
    "Missing": product_df[feature_cols].isna().sum(),
}).round(5)

product_feature_dictionary.join(product_feature_stats)

,Level,Description,Type / Unit,Min,Max,Mean,Median,SD,Missing
Feature,,,,,,,,,
name_length,Product,Number of term ids in the product name,count (terms),1.00000,78.00000,27.91326,27.00000,8.05645,0
has_brand,Product,"1 if brand_name is present, else 0",binary,0.00000,1.00000,0.92244,1.00000,0.26748,0
shop_is_missing,Product,"1 if shop_id is the -1 placeholder, else 0",binary,0.00000,1.00000,0.00850,0.00000,0.09179,0
cate_depth,Product,3 if the level-4 category repeats level 3 (sam...,ordinal (3/4),3.00000,4.00000,3.14724,3.00000,0.35434,0
brand_product_count,Brand,Catalogue products under the same brand (missi...,count (products),1.00000,100395.00000,6010.99756,444.00000,16436.88281,941660
shop_product_count,Shop,Catalogue products in the same shop (missing f...,count (products),1.00000,16179.00000,441.19403,125.00000,1215.46606,103162
cate4_product_count,Leaf category,Catalogue products in the same leaf category,count (products),1.00000,154584.00000,18091.64094,9095.00000,25053.58086,0
hist_click_count,Product,Clicks on the product across all users' histories,count (interactions),0.00000,12314.00000,0.67496,0.00000,8.73315,0
hist_cart_count,Product,Add-to-carts across all users' histories,count (interactions),0.00000,31238.00000,0.90101,0.00000,12.84739,0


### Planned Features: Query–Product & User–Product Match

The features below are planned for the final report and not computed yet. They target the main gap in the previous group's model, which never used the search query and had no features that differ between candidates in the same session.

Query and product text share the same term-id vocabulary, so overlap can be measured directly. All history-based features use `history_*` columns only, never `candidate_label_list`.

In [23]:
planned_feature_dictionary = pd.DataFrame([
    ("query_name_overlap", "Query–product", "Share of query term ids that appear in the product name", "ratio (0-1)"),
    ("query_brand_overlap", "Query–product", "Share of query term ids that appear in the brand name", "ratio (0-1)"),
    ("query_cate_overlap", "Query–product", "Share of query term ids that appear in any category name", "ratio (0-1)"),
    ("bm25_query_name", "Query–product", "BM25 relevance score between the query and the product name", "score (continuous)"),
    ("query_cate_affinity", "Query–category", "Share of past interactions after searches with the query's terms that fall in the candidate's leaf category", "ratio (0-1)"),
    ("user_cate_affinity", "User–product", "Time-decayed, action-weighted share of the user's history in the candidate's leaf category", "ratio (0-1)"),
    ("user_brand_match", "User–product", "1 if the user interacted with the candidate's brand before, else 0", "binary"),
    ("user_shop_match", "User–product", "1 if the user interacted with the candidate's shop before, else 0", "binary"),
    ("candidate_in_history", "User–product", "1 if the user interacted with this exact product before, else 0", "binary"),
], columns=["Feature", "Level", "Description", "Type / Unit"]).set_index("Feature")
planned_feature_dictionary

,Level,Description,Type / Unit
Feature,,,
query_name_overlap,Query–product,Share of query term ids that appear in the pro...,ratio (0-1)
query_brand_overlap,Query–product,Share of query term ids that appear in the bra...,ratio (0-1)
query_cate_overlap,Query–product,Share of query term ids that appear in any cat...,ratio (0-1)
bm25_query_name,Query–product,BM25 relevance score between the query and the...,score (continuous)
query_cate_affinity,Query–category,Share of past interactions after searches with...,ratio (0-1)
user_cate_affinity,User–product,"Time-decayed, action-weighted share of the use...",ratio (0-1)
user_brand_match,User–product,1 if the user interacted with the candidate's ...,binary
user_shop_match,User–product,1 if the user interacted with the candidate's ...,binary
candidate_in_history,User–product,1 if the user interacted with this exact produ...,binary
